In [1]:
import numpy as np
import pandas as pd
import json
import time
import datetime
import calendar

In [2]:
# load in psiturk data
rm1df = pd.read_json('../../data/db/exported/room1-2.8.19.json')
rm2df = pd.read_json('../../data/db/exported/room2-2.8.19.json')

# drop runs that didn't finish
rm1df = rm1df[rm1df.status != 1]
rm2df = rm2df[rm2df.status != 1]

# keep relevant columns
rm1df = rm1df[['uniqueid','datastring','beginhit','endhit','hitid','status']].reset_index(drop=True)
rm2df = rm2df[['uniqueid','datastring','beginhit','endhit','hitid','status']].reset_index(drop=True)

# format datastring as dict 
rm1df['datastring'] = rm1df['datastring'].apply(json.loads)
rm2df['datastring'] = rm2df['datastring'].apply(json.loads)

# add test room column
rm1df['testroom'] = 1
rm2df['testroom'] = 2

# concatenate dataframes
expdf = pd.concat([rm1df, rm2df], ignore_index=True)

In [3]:
expdf

,uniqueid,datastring,beginhit,endhit,hitid,status,testroom
0,debugrcDp9:debugAniFb,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-02 22:31:54.578217,2018-10-02 23:35:38.071920,debugvPYXO,3,1
1,debugIEH2T:debugDLVLJ,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 18:14:31.394716,2018-10-12 19:12:06.999050,debug7rmxU,3,1
2,debugBUnNA:debugLtZcs,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 19:17:03.503575,2018-10-12 20:10:44.883411,debugTkKFp,3,1
3,debugd1YD1:debug4FrAg,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 20:22:07.319435,2018-10-12 21:09:46.566325,debugonOYk,3,1
4,debugGaDml:debugFTHoY,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 21:28:01.305584,2018-10-12 22:16:31.057617,debugXHY6O,3,1
5,debugGVTD3:debugfpzCT,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 22:59:26.829625,2018-10-13 00:01:24.711451,debuggQ0y6,3,1
6,debugokLIG:debugalG88,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 18:28:34.378931,2018-10-13 19:15:12.977411,debugslG65,3,1
7,debugdhnfF:debug6fW93,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 19:35:43.889748,2018-10-13 20:32:37.663906,debugszpCm,3,1
8,debugHKLdw:debugk43rK,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 20:37:43.034617,2018-10-13 21:27:38.367721,debugHGwAN,3,1
9,debugnj3ww:debugAC3on,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 21:34:31.560211,2018-10-13 22:25:03.538624,debughqOmP,3,1


In [4]:
# load pre/post questionnaire responses
preqdf = pd.read_csv('../../data/google-form-data/Pre-experiment Questionnaire.csv', parse_dates=[0])
preqdf = preqdf.rename(index=str, columns={'Timestamp':'preqtime'})
postqdf = pd.read_csv('../../data/google-form-data/Post-experiment questionnaire.csv', parse_dates=[0])
postqdf = postqdf.rename(index=str, columns={'Timestamp':'postqtime'})

# exclude test runs
preqdf = preqdf.dropna(subset=['Subject ID'], inplace=False).reset_index(drop=True)
postqdf = postqdf.dropna(subset=['Subject ID'], inplace=False).reset_index(drop=True)

# convert form timestamp to POSIX time **SERIES.APPLY(DT.DT.TIMESTAMP).MULTIPLY(1000) DOES NOT WORK**
newpretimestamp = pd.Series([0]*len(preqdf['preqtime']))
for ix, val in enumerate(newpretimestamp):
    newpretimestamp[ix] = preqdf['preqtime'][ix].timestamp()*1000
preqdf['preqtime'] = newpretimestamp

newposttimestamp = pd.Series([0]*len(postqdf['postqtime']))
for ix, val in enumerate(newposttimestamp):
    newposttimestamp[ix] = postqdf['postqtime'][ix].timestamp()*1000
postqdf['postqtime'] = newposttimestamp

In [ ]:
# remove dropped subjects from google form and experiment dfs

dropids = ['MD-102218-B-04','MD-020119-A-01','MD-102318-A-01','MD-101318-A-05','MD-020119-B-01']

### for mapping between Google Forms with experiment IDs and SQLite databases with PsiTurk IDs

In [43]:
# add empty columns from pre/postquestionnaires to expdf
newcols = pd.unique(np.concatenate([i.columns.values for i in [preqdf,postqdf]]))
expdf = pd.concat([expdf, pd.DataFrame(columns=newcols)], sort=False)

turktimes = {}
preqtimes = {}

# get psiturk start times
for ix, sub in expdf.iterrows():
    turktimes[sub['datastring']['data'][0]['dateTime']] = ix
# get gform submit times
for ix, sub in preqdf.iterrows():
    preqtimes[sub['preqtime']] = ix

# for each psiturk time, find row with closest gform submission time
for turktime, rownum in turktimes.items():
    closest = preqtimes.get(turktime, preqtimes[min(preqtimes.keys(), key=lambda k: abs(k-turktime))])
    print(rownum, closest)
    
    
    


0 0
1 1
2 3
3 5
4 7
5 7
6 10
7 13
8 14
9 16
10 18
11 19
12 22
13 23
14 25
15 26
16 27
17 29
18 31
19 32
20 33
21 35
22 38
23 39
24 40
25 41
26 44
27 47
28 49
29 51
30 51
31 54
32 56
33 58
34 61
35 62
36 64
37 65
38 67
39 70
40 72
41 73
42 74
43 77
44 78
45 79
46 81
47 82
48 84
49 85
50 88
51 90
52 90
53 92
54 93
55 95
56 98
57 99
58 101
59 103
60 104
61 106
62 107
63 109
64 110
65 0
66 4
67 6
68 7
69 10
70 13
71 14
72 16
73 19
74 23
75 25
76 27
77 31
78 35
79 38
80 40
81 44
82 45
83 48
84 49
85 51
86 52
87 56
88 58
89 60
90 62
91 65
92 65
93 68
94 70
95 72
96 74
97 77
98 81
99 84
100 84
101 88
102 90
103 92
104 95
105 100
106 102
107 105
108 108


In [41]:
rownum

108

In [40]:
turktimes

{1538358760289: 65,
 1538519554757: 0,
 1539368162836: 1,
 1539371956776: 2,
 1539372566510: 66,
 1539375821845: 3,
 1539376317256: 67,
 1539379785466: 4,
 1539379962852: 68,
 1539385290305: 5,
 1539455406718: 6,
 1539455715708: 69,
 1539459433664: 7,
 1539459605736: 70,
 1539462690137: 71,
 1539463137045: 8,
 1539466594814: 9,
 1539467233097: 72,
 1539470310153: 10,
 1539473801650: 73,
 1539475256939: 11,
 1539715493736: 12,
 1539718733955: 13,
 1539719119292: 74,
 1539722660369: 14,
 1539722936487: 75,
 1539726182917: 15,
 1539730420132: 16,
 1539730837786: 76,
 1539874472831: 17,
 1539882724921: 18,
 1539883383447: 77,
 1539892919172: 19,
 1539905903265: 20,
 1539963274958: 21,
 1539963302559: 78,
 1539973918934: 22,
 1539973956520: 79,
 1539977238916: 23,
 1539980678664: 80,
 1539981844351: 24,
 1539989239921: 25,
 1540054720879: 26,
 1540055554764: 81,
 1540058839864: 82,
 1540062768079: 27,
 1540063248489: 83,
 1540066649585: 28,
 1540066957684: 84,
 1540070647428: 29,
 154007099